# Chapter 15. Operating on Data in Pandas

## 15.1 Ufuncs: Index Preservation

When applying NumPy's ufunc to a Pandas object, the index and column labels are preserved.

In [22]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# to Series applying ufunc
ser = pd.Series(rng.integers(0, 10, 4))
print("Original Series:") 
print(ser)

print("\nexp(Series):")
print(np.exp(ser))

Original Series:
0    0
1    7
2    6
3    4
dtype: int64

exp(Series):
0       1.000000
1    1096.633158
2     403.428793
3      54.598150
dtype: float64


In [23]:
# to DataFrame applying ufunc
df = pd.DataFrame(rng.integers(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
print("Original DataFrame:")
print(df)

print("\nsin(DataFrame * π/4):")
print(np.sin(df * np.pi / 4))

Original DataFrame:
   A  B  C  D
0  4  8  0  6
1  2  0  5  9
2  7  7  7  7

sin(DataFrame * π/4):
              A             B         C         D
0  1.224647e-16 -2.449294e-16  0.000000 -1.000000
1  1.000000e+00  0.000000e+00 -0.707107  0.707107
2 -7.071068e-01 -7.071068e-01 -0.707107 -0.707107


💡 NumPy의 모든 ufunc는 Pandas 객체에서도 그대로 작동하며, 인덱스 정보를 유지한다.

## 15.2 Ufuncs: Index Alignment - Very Very important!

Pandas의 가장 강력한 기능 중 하나: 두 Series/DataFrame 간 연산 시 인덱스를 자동으로 정렬

Index Alignment in Series

In [24]:
# Creation two Series
area = pd.Series({'Alaska': 1723337, 'Texas': 695662, 
                  'California': 423967}, name='area')
population = pd.Series({'California': 39538223, 'Texas': 29145505, 
                        'Florida': 21538187}, name='population')

print("area:")
print(area)

print("\npopulation:")
print(population)

area:
Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64

population:
California    39538223
Texas         29145505
Florida       21538187
Name: population, dtype: int64


In [25]:
# 인구 밀도
density = population / area
print(density)

Alaska              NaN
California    93.257784
Florida             NaN
Texas         41.896072
dtype: float64


In [26]:
# Check union index
print(area.index.union(population.index))

Index(['Alaska', 'California', 'Florida', 'Texas'], dtype='str')


Using fill_value

In [27]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])

print("A + B (main):")
print(A + B)

print("\nA.add(B, fill_value=0):")
print(A.add(B, fill_value=0))

A + B (main):
0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

A.add(B, fill_value=0):
0    2.0
1    5.0
2    9.0
3    5.0
dtype: float64


Index Alignment in DataFrames

In [28]:
# Two DataFrames Creation
A = pd.DataFrame(rng.integers(0, 20, (2, 2)),
                 columns=['a', 'b'])
print("A:")
print(A)

B = pd.DataFrame(rng.integers(0, 10, (3, 3)),
                 columns=['b', 'a', 'c'])
print("\nB:")
print(B)

print("\nA + B:")
print(A + B)

A:
    a  b
0  10  2
1  16  9

B:
   b  a  c
0  5  3  1
1  9  7  6
2  4  8  5

A + B:
      a     b   c
0  13.0   7.0 NaN
1  23.0  18.0 NaN
2   NaN   NaN NaN


💡 DataFrame도 마찬가지로 행 인덱스와 열 인덱스가 모두 정렬됩니다.

In [29]:
# using fill_value
print(A.add(B, fill_value=A.values.mean()))

       a      b      c
0  13.00   7.00  10.25
1  23.00  18.00  15.25
2  17.25  13.25  14.25


Mapping Python Operators and Pandas Methods
| Python operation | Pandas method |
|---------------|---------------|
| `+` | `.add()` |
| `-` | `.sub()`, `.subtract()` |
| `*` | `.mul()`, `.multiply()` |
| `/` | `.div()`, `.divide()`, `.truediv()` |
| `//` | `.floordiv()` |
| `%` | `.mod()` |
| `**` | `.pow()` |

## 15.3 Ufuncs: Operations Between DataFrame and Series

In [30]:
# 3X4 array creation
A = rng.integers(10, size=(3, 4))
print("A:")
print(A)

df = pd.DataFrame(A, columns=['Q', 'R', 'S', 'T'])
print("\ndf:")
print(df)

A:
[[4 4 2 0]
 [5 8 0 8]
 [8 2 6 1]]

df:
   Q  R  S  T
0  4  4  2  0
1  5  8  0  8
2  8  2  6  1


Default: Row direction operation (broadcasting)

In [ ]:
# 첫 번째 행을 각 행에서 빼기
print(df - df.iloc[0])

   Q  R  S  T
0  0  0  0  0
1  1  4 -2  8
2  4 -2  4  1


Column direction operation (axis specified)

In [ ]:
# 'R' 렬 값을 각 렬에서 빼기
print(df.subtract(df['R'], axis=0))

   Q  R  S  T
0  0  0 -2 -4
1 -3  0 -8  0
2  6  0  4 -1
